[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/17_dropout_solution.ipynb)

# 🟢 Solution: Implement Dropout（参考版）

## 📋 题目描述

**难度：** Easy

实现 Dropout（nn.Module）。

Dropout 在训练时以概率 p 随机将元素置零，并将存活元素缩放 `1/(1-p)` 以保持期望值不变。推理时为恒等映射。

**签名:** `MyDropout(p=0.5)`（nn.Module）

**前向传播:** `forward(x) -> Tensor`
- `x` — 任意形状的输入张量

**返回:** 应用 dropout 后的张量（训练）或原始输入（推理）

**约束:**
- 训练模式：以概率 p 置零，缩放 `1/(1-p)`
- 推理模式：返回原始输入

**提示：** 训练：`mask = (rand_like(x) > p).float()` → `x * mask / (1-p)`
推理：直接返回 `x`


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ SOLUTION

class MyDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        # ---- Step 1: 保存丢弃概率 ----
        # p 是每个元素被置零的概率
        # p=0.5 表示平均有一半的元素会被丢弃
        self.p = p

    def forward(self, x):
        # ---- Step 2: 推理模式直接返回 ----
        # self.training 是 nn.Module 的内置属性
        # model.eval() 时 self.training = False，model.train() 时 = True
        # 推理时不做任何随机操作，保证输出确定性
        if not self.training or self.p == 0:
            return x

        # ---- Step 3: 训练模式生成掩码 ----
        # torch.rand_like(x) 生成与 x 同 shape 的 [0,1) 均匀分布随机数
        # > self.p：大于 p 的位置为 True（保留），否则为 False（丢弃）
        # 每个位置保留的概率 = 1 - p
        mask = (torch.rand_like(x) > self.p).float()

        # ---- Step 4: 缩放补偿 ----
        # 为什么除以 (1-p)？
        # 假设 x 的某个元素值为 v，期望 E[输出] = v * (1-p) / (1-p) = v
        # 不缩放的话，训练时激活值的期望会比推理时小 (1-p) 倍
        # 这就是"inverted dropout"——训练时缩放，推理时不用管
        return x * mask / (1 - self.p)


In [ ]:
drop = MyDropout(p=0.5)
x = torch.ones(10)
drop.train()
print('Train:', drop(x))
drop.eval()
print('Eval: ', drop(x))


In [ ]:
from torch_judge import check
check('dropout')


## 📝 核心思路总结

1. **训练 vs 推理**：训练时随机置零 + 缩放，推理时恒等映射
2. **掩码生成**：`rand_like(x) > p` 生成伯努利掩码，保留概率 = 1-p
3. **Inverted Dropout**：训练时除以 (1-p) 保持期望不变，推理时零开销
4. **防过拟合原理**：类似集成学习 + 强制特征冗余
